# Airline Ticket Price Prediction

## Practical 1 — Web scraping demonstration
This small example shows how scraping works on a public practice website. It collects book titles, **not airline prices**. Do not scrape airline websites unless their rules allow it.

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

url = "https://books.toscrape.com/"
page = requests.get(url)
soup = BeautifulSoup(page.text, "html.parser")

titles = []
for book in soup.find_all("h3"):
    titles.append(book.find("a")["title"])

print(titles[:10])

## Load the airline dataset

In [ ]:
from google.colab import files

uploaded = files.upload()  # Choose the airline CSV file
file_name = list(uploaded.keys())[0]

df = pd.read_csv(file_name)

print("Rows and columns:", df.shape)
print(df.head())

## Practical 2A — Data cleaning and descriptive statistics

In [ ]:
# See missing values and duplicate rows
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

# Remove duplicate rows
df = df.drop_duplicates()

# Remove rows where ticket price is missing
df = df.dropna(subset=["price"])

# Basic statistics for ticket price
print("Mean:", df["price"].mean())
print("Median:", df["price"].median())
print("Mode:", df["price"].mode()[0])
print("Variance:", df["price"].var())
print("Standard deviation:", df["price"].std())
print("Quartiles:")
print(df["price"].quantile([0.25, 0.50, 0.75]))

# Summary statistics
print(df.describe())

## Practical 2B — Data visualization

In [ ]:
import matplotlib.pyplot as plt

# 1. Histogram
plt.hist(df["price"], bins=30)
plt.title("Distribution of Ticket Prices")
plt.xlabel("Ticket price")
plt.ylabel("Number of flights")
plt.show()

# 2. Box plot
plt.boxplot(df["price"])
plt.title("Ticket Price Outliers")
plt.ylabel("Ticket price")
plt.show()

# 3. Bar chart: average price by airline
df.groupby("airline")["price"].mean().plot(kind="bar")
plt.title("Average Price by Airline")
plt.ylabel("Average price")
plt.show()

# 4. Pie chart: number of flights by airline
df["airline"].value_counts().plot(kind="pie", autopct="%1.1f%%")
plt.title("Flights by Airline")
plt.ylabel("")
plt.show()

# 5. Scatter plot: days left vs price
plt.scatter(df["days_left"], df["price"], alpha=0.3)
plt.title("Days Left vs Ticket Price")
plt.xlabel("Days left before departure")
plt.ylabel("Ticket price")
plt.show()

# 6. Line chart: average price by days left
df.groupby("days_left")["price"].mean().sort_index().plot(kind="line")
plt.title("Average Ticket Price by Days Left")
plt.xlabel("Days left")
plt.ylabel("Average price")
plt.show()

## Practical 3 — Probability and sampling

In [ ]:
import numpy as np

# Probability that a randomly selected ticket costs more than the mean
mean_price = df["price"].mean()
probability = (df["price"] > mean_price).mean()
print("Probability price is above the mean:", probability)

# Random sample of 10 flights
random_sample = df.sample(n=10, random_state=1)
print(random_sample[["airline", "price"]])

# Sample ticket prices and view their distribution
plt.hist(random_sample["price"], bins=5)
plt.title("Prices in a Random Sample")
plt.xlabel("Ticket price")
plt.show()

## Practical 4 — Central Limit Theorem and point estimation

In [ ]:
# Take 100 samples and calculate each sample's mean price
sample_means = []

for i in range(100):
    sample = df["price"].sample(n=30, replace=True, random_state=i)
    sample_means.append(sample.mean())

print("Population mean estimate:", df["price"].mean())
print("Mean of sample means:", np.mean(sample_means))

plt.hist(sample_means, bins=15)
plt.title("Central Limit Theorem: Sample Means")
plt.xlabel("Sample mean price")
plt.ylabel("Frequency")
plt.show()

## Practical 5 — T-test
We compare ticket prices for nonstop and connecting flights. The code checks the dataset's `stops` values and assumes `zero` means nonstop, as in the common Kaggle dataset.

In [ ]:
from scipy.stats import ttest_ind

print(df["stops"].value_counts())

# Separate nonstop and connecting flight prices
nonstop = df[df["stops"].astype(str).str.lower() == "zero"]["price"]
connecting = df[df["stops"].astype(str).str.lower() != "zero"]["price"]

# Welch's independent t-test
t_value, p_value = ttest_ind(nonstop, connecting, equal_var=False)

print("Average nonstop price:", nonstop.mean())
print("Average connecting price:", connecting.mean())
print("T statistic:", t_value)
print("P value:", p_value)

if p_value < 0.05:
    print("There is a statistically significant difference in average prices.")
else:
    print("There is not enough evidence of a difference in average prices.")

## Practical 7 — Multiple Linear Regression
To keep the code easy to explain, this model uses three numeric predictors: duration, days left, and number of stops. The target is ticket price. This is multiple linear regression because it uses more than one independent variable.

The common Kaggle dataset stores duration in text such as `2h 50m` and stops as words, so we convert them first.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

# Convert duration such as '2h 50m' into minutes
def convert_duration(value):
    hours = 0
    minutes = 0
    parts = str(value).split()
    for part in parts:
        if "h" in part:
            hours = int(part.replace("h", ""))
        if "m" in part:
            minutes = int(part.replace("m", ""))
    return hours * 60 + minutes

df["duration_minutes"] = df["duration"].apply(convert_duration)

# Convert stop categories into numbers
df["stops_number"] = df["stops"].map({
    "zero": 0,
    "one": 1,
    "two_or_more": 2
})

# Keep the columns needed for this simple model
model_data = df[["duration_minutes", "days_left", "stops_number", "price"]].dropna()

# X = input variables; y = price we want to predict
X = model_data[["duration_minutes", "days_left", "stops_number"]]
y = model_data["price"]

# Split data into training and testing parts
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict ticket prices
predictions = model.predict(X_test)

print("Model coefficients:", model.coef_)
print("Model intercept:", model.intercept_)
print("Mean Absolute Error:", mean_absolute_error(y_test, predictions))
print("R2 score:", r2_score(y_test, predictions))

In [ ]:
!pip -q install streamlit scikit-learn pandas numpy

# Create app.py from the Streamlit app code in the project files.
# In Google Colab, use the next cell to download the app file from this notebook's folder,
# or download the ready-made app.py and requirements.txt provided with the project ZIP.
print('Streamlit installed. Streamlit apps are best run locally using: streamlit run app.py')